# 2 · plot — contact-prediction accuracy

Draws [`2_make_rprecision_data.ipynb`](2_make_rprecision_data.ipynb)'s table as one panel per
protein class. Edit `PREDICTORS` to choose which predictors appear and what they are called; the
dataset holds all of them.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
DATASET = "2_rprecision"
DPI = 300
# published name -> the label to draw. Membership is what matters; the order below is the draw
# order. `MarinFold` is #232's step-363000 checkpoint, the best model trained on decontaminated
# data and the one Helico's contact arm conditions on. Its eval-test rows are not #245's — #232
# left that split unscored, so `score_foldbench_rollouts.py` scored all 333 monomers with it and
# the make notebook measures that pipeline against #245's before joining the two.
# The #199 cooldown is deliberately absent: its training corpus was never filtered against
# FoldBench, so its number is not a claim we make. It is still in the dataset (#245 published it)
# and part 4 of the exploration notebook still contrasts the two — it simply is not drawn here.
# The sweep final (`#232 m2-p06 (decontaminated)`) is in the dataset too, as the pipeline control.
#
# Order is FIXED, top to bottom, and is not a ranking: it is a ladder of how much evolutionary
# information each predictor gets. MarinFold and Protenix-v2 SS see one sequence; ESMFold and
# ESMFold2 see a protein language model trained on many; Protenix-v2 MSA sees the alignment
# itself. Both panels draw it identically, so a bar can be compared across panels by position.
PREDICTORS = {
    "#232 m2-p06 step-363000 (decontaminated)": "MarinFold",
    "Protenix-v2 single-seq": "Protenix-v2 SS",
    "ESMFold": "ESMFold",
    "ESMFold2": "ESMFold2",
    "Protenix-v2 + MSA": "Protenix-v2 MSA",
}
HIGHLIGHT = lambda predictor: predictor.startswith("#")   # noqa: E731 — drawn in the accent colour
metadata = figlib.describe(DATASET)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

figlib.figure_style(DPI)
summary = pd.read_csv(figlib.require(DATASET, "summary.csv") / "summary.csv")
ACCENT, NEUTRAL = "#C44E52", "#7A8DA6"


#: Width of a panel that will sit two-up in a 6.5 in text column — see figure 3's note.
PANEL_WIDTH = 3.25
#: Which proteins a panel is about, written inside the panel so it keeps its identity if the
#: panel is lifted out of the figure.
CLASS_LABELS = {"natural": "FoldBench natural monomers", "designed": "FoldBench de novo designs"}


def panel(frame, xlabel, name, highlight, protein_class=None):
    """One horizontal bar per predictor, in PREDICTORS order, with its bootstrap interval."""
    # barh draws the first row at the bottom, so reverse to read PREDICTORS top to bottom.
    order = [p for p in reversed(list(PREDICTORS)) if p in set(frame.predictor)]
    frame = frame.set_index("predictor").reindex(order).reset_index()
    figure, axis = plt.subplots(figsize=(PANEL_WIDTH, 0.26 * len(frame) + 0.85),
                                layout="constrained")
    axis.barh(frame.label, frame.value, height=0.6,
              color=[ACCENT if highlight(row) else NEUTRAL for row in frame.itertuples()],
              xerr=[frame.value - frame.ci_low, frame.ci_high - frame.value],
              error_kw=dict(ecolor="0.25", lw=0.9, capsize=2.5))
    for y, row in enumerate(frame.itertuples()):
        axis.text(row.ci_high + 0.02, y, f"{row.value:.2f}", va="center", fontsize=7.5,
                  color="0.25")
    axis.set(xlabel=xlabel, xlim=(0, 1.06))
    axis.grid(axis="x", alpha=0.25, lw=0.6)
    axis.set_axisbelow(True)
    if protein_class is not None:
        figure.text(0.005, 0.005, CLASS_LABELS.get(protein_class, protein_class),
                    ha="left", va="bottom", fontsize=8, color="0.35")
    figlib.save_figure(figure, name, DPI)
    plt.show()
    return frame


for protein_class in summary.protein_class.unique():
    frame = summary[(summary.protein_class == protein_class)
                    & summary.predictor.isin(PREDICTORS)].copy()
    frame["label"] = frame.predictor.map(PREDICTORS)
    print(f"--- {protein_class} · n={int(frame.n.iloc[0])} proteins ---")
    drawn = panel(frame, "R-precision", f"rprecision_{protein_class}",
                  lambda row: HIGHLIGHT(row.predictor), protein_class)
    print(drawn[["label", "n", "value", "ci_low", "ci_high"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))